# AI Product Discovery with RAG (Notebook Structure)

This notebook is the **orchestrator** for your repo-based implementation:

- `src/` holds core pipeline logic (cleaning, embeddings, FAISS, ranking, RAG, UI)
- `notebooks/` demonstrates the end-to-end workflow with plots + examples
- `data/` and `artifacts/` are **gitignored**

> **Colab tip**: If running in Colab, clone your repo and run from the repo root.


## 1) Setup & Imports


In [ ]:
# Ensure repo root is on sys.path so `import src.*` works
import os, sys
from pathlib import Path

cwd = Path.cwd()

# If running from repo root, keep it; if from notebooks/, go up one level
if (cwd / 'src').exists():
    repo_root = cwd
elif (cwd.name == 'notebooks') and ((cwd.parent / 'src').exists()):
    repo_root = cwd.parent
else:
    # Fallback: search upwards for a folder containing `src` and `requirements.txt`
    candidates = [cwd, *cwd.parents]
    repo_root = next((p for p in candidates if (p / 'src').exists()), cwd)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)
print('CWD:', cwd)
print('sys.path[0]:', sys.path[0])


In [1]:
# (Colab) If you cloned your repo, run this cell from the repo root.
# Example:
# !git clone git@github.com:<YOU>/<REPO>.git
# %cd <REPO>

# Install dependencies (Colab-only). Locally, use: pip install -r requirements.txt
# !pip -q install -r requirements.txt

import os, re, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repo modules
from src.config import EMBED_MODEL, LLM_MODEL, ARTIFACT_DIR
from src.data import load_data

# Uncomment as you implement these modules
# from src.embeddings import embed_texts
# from src.index import build_index
# from src.ranker import hybrid_score
# from src.rag import build_prompt

os.makedirs(ARTIFACT_DIR, exist_ok=True)
print("EMBED_MODEL:", EMBED_MODEL)
print("LLM_MODEL:", LLM_MODEL)
print("ARTIFACT_DIR:", ARTIFACT_DIR)


ModuleNotFoundError: No module named 'src'

## 2) Load CSV + Inspect


In [ ]:
# Path options:
# - In Colab: upload to /content, or mount Google Drive
# - Locally: point to your local file

CSV_PATH = "data/company_reviews.csv"  # TODO: change to your file path

df_raw = load_data(CSV_PATH)
print("Shape:", df_raw.shape)
display(df_raw.head(3))

# Quick schema + missingness
col_info = pd.DataFrame({
    "col": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "missing_%": [df_raw[c].isna().mean()*100 for c in df_raw.columns],
}).sort_values("missing_%", ascending=False)

display(col_info)


## 3) Cleaning & Parsing


In [ ]:
# TODO: Prefer moving these into src/data.py as you finalize.
# For now, implement minimal inline cleaning so notebook runs.

def to_float(x):
    try:
        if pd.isna(x): return np.nan
        s = str(x).strip().replace("%","")
        return float(s)
    except Exception:
        return np.nan

def parse_listish(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    s = str(x).strip()
    if not s: return []
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return [str(t).strip() for t in obj if str(t).strip()]
        except Exception:
            pass
    for sep in [";", "|", ","]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]
    return [s]

def parse_salary(x):
    # Returns (min, max, median) from strings like "$90k-$120k"
    if pd.isna(x): 
        return (np.nan, np.nan, np.nan)
    s = str(x).lower().replace("$","").replace("usd","").strip()
    s = s.replace("k","000")
    nums = re.findall(r"([0-9][0-9,]*)", s)
    nums = [float(n.replace(",","")) for n in nums] if nums else []
    if len(nums) == 0:
        return (np.nan, np.nan, np.nan)
    if len(nums) == 1:
        return (nums[0], nums[0], nums[0])
    mn, mx = min(nums), max(nums)
    return (mn, mx, (mn+mx)/2)

df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

# Numeric columns (edit as needed)
for c in ["rating", "happiness", "ceo_approval", "interview_difficulty", "interview_experience"]:
    if c in df.columns:
        df[c] = df[c].apply(to_float)

# List-like columns (edit as needed)
for c in ["roles", "locations", "ratings"]:
    if c in df.columns:
        df[c] = df[c].apply(parse_listish)

# Salary parsing
if "salary" in df.columns:
    parsed = df["salary"].apply(parse_salary)
    df["salary_min"] = [t[0] for t in parsed]
    df["salary_max"] = [t[1] for t in parsed]
    df["salary_median"] = [t[2] for t in parsed]

# Build doc text
text_cols = [c for c in ["description", "reviews"] if c in df.columns]
df["doc_text"] = df[text_cols].fillna("").astype(str).agg("\n".join, axis=1).str.replace(r"\s+"," ", regex=True).str.strip()

df_clean = df
print("Clean shape:", df_clean.shape)
display(df_clean.head(3))


## 4) Quick EDA


In [ ]:
# Keep EDA tight (4–6 plots max). Add/remove columns based on your data.

def hist_if_exists(col):
    if col in df_clean.columns and df_clean[col].notna().any():
        plt.figure()
        df_clean[col].dropna().plot(kind="hist", bins=30, title=f"Distribution: {col}")
        plt.xlabel(col)
        plt.show()

for col in ["rating", "happiness", "ceo_approval", "salary_median"]:
    hist_if_exists(col)

# Top roles / locations (optional)
def top_k_listcol(col, k=15):
    if col not in df_clean.columns:
        return None
    items = []
    for lst in df_clean[col]:
        items.extend(lst if isinstance(lst, list) else [])
    s = pd.Series(items).value_counts().head(k)
    plt.figure()
    s.sort_values().plot(kind="barh", title=f"Top {k}: {col}")
    plt.show()
    return s

_ = top_k_listcol("roles", k=15)
_ = top_k_listcol("locations", k=15)


## 5) Build Retrieval Documents


In [ ]:
# Define retrieval unit: start with 1 row -> 1 document.

NAME_COL = "name" if "name" in df_clean.columns else df_clean.columns[0]

def build_docs(df):
    docs = []
    for i, row in df.iterrows():
        meta = {"row_index": int(i), "company": str(row.get(NAME_COL, i))}
        # Keep a few structured signals for ranking/filters
        for c in ["rating","happiness","ceo_approval","salary_median","salary_min","salary_max","roles","locations"]:
            if c in df.columns:
                meta[c] = row.get(c)
        docs.append({"id": str(i), "text": str(row.get("doc_text","")), "meta": meta})
    return docs

docs = build_docs(df_clean)
print("Docs:", len(docs))
print("Example meta:", docs[0]["meta"])
print("Example text preview:", docs[0]["text"][:240])


## 6) Embeddings


In [ ]:
# TODO: move embedding logic into src/embeddings.py and import embed_texts.
# Example:
# texts = [d["text"] or " " for d in docs]
# emb = embed_texts(texts, EMBED_MODEL)
# np.save(os.path.join(ARTIFACT_DIR, "embeddings.npy"), emb)

raise NotImplementedError("Implement src/embeddings.py -> embed_texts(), then uncomment this section.")


## 7) FAISS Index


In [ ]:
# TODO: move FAISS logic into src/index.py and import build_index.
# Example:
# index = build_index(emb)
# import faiss
# faiss.write_index(index, os.path.join(ARTIFACT_DIR, "faiss.index"))

raise NotImplementedError("Implement src/index.py -> build_index(), then uncomment this section.")


## 8) Retrieval + Filters Demo


In [ ]:
# TODO:
# - Implement search(query, top_k, filters)
# - Use embedding model + FAISS to retrieve
# - Then filter by metadata (location/role/min_salary/min_rating)

# Suggested signature:
# def search(query: str, top_k: int = 10, filters: dict | None = None) -> list[dict]:
#     ...

raise NotImplementedError("Implement retrieval (suggest: src/retrieval.py), then run demo queries here.")


## 9) Hybrid Ranking with User Priorities


In [ ]:
# TODO:
# - Normalize structured columns (salary_median, rating, happiness, etc.)
# - Combine with semantic similarity using weights from user priorities
# - Output a ranked table

# Example priorities:
# priorities = {"salary_median":0.4, "rating":0.3, "happiness":0.3}

raise NotImplementedError("Implement ranker (suggest: src/ranker.py), then run ranking experiments here.")


## 10) RAG Answer Generation (Grounded)


In [ ]:
# TODO:
# - Load a small instruct model (LLM_MODEL)
# - Format retrieved context with citations (row=IDX)
# - Generate answer that ONLY uses provided context
# - Return Top 3 recommendations + reasons with citations

raise NotImplementedError("Implement RAG (suggest: src/rag.py), then generate grounded answers here.")


## 11) Side-by-Side Comparison


In [ ]:
# TODO:
# - Select top N or user-selected companies
# - Create a structured comparison table
# - Ask LLM for a comparison summary (salary-first vs culture/WLB-first) with citations

raise NotImplementedError("Implement comparison utilities (suggest: src/compare.py), then run comparison here.")


## 12) Lightweight Evaluation


In [ ]:
# TODO:
# - Create a small set of test queries (10–30)
# - Log retrieval results and qualitative judgment
# - Optional: weak supervision Recall@K (if you can derive a 'ground truth' from metadata)

raise NotImplementedError("Implement eval (suggest: src/eval.py), then summarize metrics here.")


## 13) Gradio Demo UI


In [ ]:
# TODO:
# - Implement `build_app()` in src/ui.py returning a gradio Blocks app
# - UI should support:
#   - query box
#   - sliders for priorities
#   - optional role/location filters
#   - results table + RAG explanation

# Example:
# from src.ui import build_app
# demo = build_app()
# demo.launch(share=True)

raise NotImplementedError("Implement src/ui.py -> build_app(), then launch here.")


## 14) Save Artifacts + Local Run Notes


In [ ]:
# Optional artifact bundling:
# import shutil
# shutil.make_archive("artifacts_bundle", "zip", ARTIFACT_DIR)
# print("Created artifacts_bundle.zip")


## 15) Resume Bullets & Next Steps

- Built a RAG-based product discovery engine with FAISS semantic search over company reviews + metadata.
- Designed a hybrid ranking model combining embedding similarity with structured signals and user-weighted priorities.
- Deployed an interactive Gradio UI enabling natural-language search, filtering, and AI side-by-side comparisons with citations.
